In [ ]:
import random

MAX_COMPONENTES = 3
MAX_CONEXIONES_EXTRA = 4

# ==========================
# ESTRUCTURAS BASE
# ==========================

class Celda:
    """Celda del laberinto. __slots__ reduce memoria y acelera el acceso."""
    __slots__ = ("paredSuperior", "paredInferior", "paredIzquierda", "paredDerecha")

    def __init__(self):
        self.paredSuperior = True
        self.paredInferior = True
        self.paredIzquierda = True
        self.paredDerecha = True


# (delta_fila, delta_col, pared_propia, pared_de_la_celda_vecina)
DIRS = (
    (-1, 0, "paredSuperior", "paredInferior"),
    (1, 0, "paredInferior", "paredSuperior"),
    (0, -1, "paredIzquierda", "paredDerecha"),
    (0, 1, "paredDerecha", "paredIzquierda"),
)


def _abrir(lab, i, j, d):
    di, dj, propia, opuesta = DIRS[d]
    setattr(lab[i][j], propia, False)
    setattr(lab[i + di][j + dj], opuesta, False)


# ==========================
# GENERACIÓN DE UN BOSQUE ALEATORIO CON KRUSKAL
# ==========================

class ConjuntosDisjuntos:
    """Union-Find con compresión de caminos y unión por rango."""

    def __init__(self, cantidad):
        self.padre = list(range(cantidad))
        self.rango = [0] * cantidad

    def buscar(self, elemento):
        while elemento != self.padre[elemento]:
            self.padre[elemento] = self.padre[self.padre[elemento]]
            elemento = self.padre[elemento]
        return elemento

    def unir(self, primero, segundo):
        raiz_1 = self.buscar(primero)
        raiz_2 = self.buscar(segundo)
        if raiz_1 == raiz_2:
            return False

        if self.rango[raiz_1] < self.rango[raiz_2]:
            raiz_1, raiz_2 = raiz_2, raiz_1
        self.padre[raiz_2] = raiz_1
        if self.rango[raiz_1] == self.rango[raiz_2]:
            self.rango[raiz_1] += 1
        return True


def generarBosqueAleatorio(lab, filas, columnas):
    """
    Construye con Kruskal un bosque de entre 1 y 3 componentes. Después
    añade hasta 4 conexiones internas sin unir componentes diferentes.
    No busca la salida ni determina si el laberinto tiene solución.
    """
    conexiones = []
    for i in range(filas):
        for j in range(columnas):
            if i < filas - 1:
                conexiones.append((i, j, 1, i + 1, j))
            if j < columnas - 1:
                conexiones.append((i, j, 3, i, j + 1))
    random.shuffle(conexiones)

    total_celdas = filas * columnas
    componentes_objetivo = random.randint(1, min(MAX_COMPONENTES, total_celdas))
    componentes = total_celdas
    conjuntos = ConjuntosDisjuntos(total_celdas)

    for i, j, d, ni, nj in conexiones:
        if componentes == componentes_objetivo:
            break
        actual = i * columnas + j
        vecina = ni * columnas + nj
        if conjuntos.unir(actual, vecina):
            _abrir(lab, i, j, d)
            componentes -= 1

    # Estas paredes crean ciclos, pero nunca conectan dos componentes distintos.
    candidatas = []
    for i in range(filas):
        for j in range(columnas):
            for d in (1, 3):  # inferior y derecha, sin duplicar paredes
                di, dj, pared, _ = DIRS[d]
                ni, nj = i + di, j + dj
                if not (0 <= ni < filas and 0 <= nj < columnas):
                    continue
                if not getattr(lab[i][j], pared):
                    continue
                actual = i * columnas + j
                vecina = ni * columnas + nj
                if conjuntos.buscar(actual) == conjuntos.buscar(vecina):
                    candidatas.append((i, j, d))

    random.shuffle(candidatas)
    cantidad = random.randint(0, min(MAX_CONEXIONES_EXTRA, len(candidatas)))
    for i, j, d in candidatas[:cantidad]:
        _abrir(lab, i, j, d)


# ==========================
# ENTRADA, SALIDA Y CONEXIONES ALEATORIAS
# ==========================

def establecerEntradaSalida(lab, filas, columnas):
    """Coloca aleatoriamente la entrada arriba y la salida abajo."""
    inicio = (0, random.randrange(columnas))
    fin = (filas - 1, random.randrange(columnas))

    lab[inicio[0]][inicio[1]].paredSuperior = False
    lab[fin[0]][fin[1]].paredInferior = False
    return inicio, fin


def crearLaberinto(filas, columnas):
    """Crea un laberinto aleatorio sin comprobar si tiene solución."""
    if not isinstance(filas, int) or not isinstance(columnas, int):
        raise TypeError("Las dimensiones deben ser números enteros.")
    if filas <= 0 or columnas <= 0:
        raise ValueError("Las dimensiones deben ser mayores que cero.")

    lab = [[Celda() for _ in range(columnas)] for _ in range(filas)]
    generarBosqueAleatorio(lab, filas, columnas)
    inicio, fin = establecerEntradaSalida(lab, filas, columnas)
    return lab, inicio, fin


# ==========================
# IMPRESIÓN
# ==========================

def imprimirLaberinto(lab, filas, columnas, inicio, fin):
    lineas = []
    for i in range(filas):
        borde = ["+"]
        cuerpo = []
        for j in range(columnas):
            c = lab[i][j]
            borde.append("---" if c.paredSuperior else "   ")
            borde.append("+")
            cuerpo.append("|" if c.paredIzquierda else " ")
            if (i, j) == inicio:
                cuerpo.append(" S ")
            elif (i, j) == fin:
                cuerpo.append(" E ")
            else:
                cuerpo.append("   ")
        cuerpo.append("|" if lab[i][columnas - 1].paredDerecha else " ")
        lineas.append("".join(borde))
        lineas.append("".join(cuerpo))

    base = ["+"]
    for j in range(columnas):
        base.append("---" if lab[filas - 1][j].paredInferior else "   ")
        base.append("+")
    lineas.append("".join(base))
    print("\n".join(lineas))


# ==========================
# PROGRAMA PRINCIPAL
# ==========================

if __name__ == "__main__":
    filas = int(input("Ingrese la cantidad de filas: "))
    columnas = int(input("Ingrese la cantidad de columnas: "))

    lab, inicio, fin = crearLaberinto(filas, columnas)

    print(f"\nEntrada: {inicio} | Salida: {fin}")
    print()

    imprimirLaberinto(lab, filas, columnas, inicio, fin)



ALGORITMO DETERMINISTA

In [31]:
# ==========================================================
# BLOQUE 2: AGENTE DETERMINISTA (DFS con backtracking)
# Recibe el laberinto generado en el Bloque 1.
# ==========================================================

NOMBRES = ("arriba", "abajo", "izquierda", "derecha")


class Entorno:
    """Interfaz de percepción que evita que el agente manipule el laberinto."""

    def __init__(self, lab, filas, columnas):
        self._lab, self._f, self._c = lab, filas, columnas

    def entradas(self):
        """Aberturas en el borde superior (puntos de arranque)."""
        return [(0, j) for j in range(self._c) if not self._lab[0][j].paredSuperior]

    def percibir(self, pos):
        """
        Devuelve las aberturas visibles desde 'pos' en el orden fijo de DIRS.
        Cada percepción tiene la forma (direccion, tipo, destino), donde tipo
        es 'celda' para un pasillo interno o 'borde' para una abertura exterior.
        """
        i, j = pos
        celda = self._lab[i][j]
        vistas = []
        for d, (di, dj, pared, _) in enumerate(DIRS):
            if getattr(celda, pared):
                continue
            ni, nj = i + di, j + dj
            if 0 <= ni < self._f and 0 <= nj < self._c:
                vistas.append((d, "celda", (ni, nj)))
            else:
                vistas.append((d, "borde", None))
        return vistas


class AgenteLaberinto:
    """
    Enumera mediante DFS iterativo todos los caminos simples desde la entrada
    hasta cualquier abertura exterior. El orden fijo de DIRS hace que su política
    sea determinista. Una celda puede aparecer en caminos diferentes, pero nunca
    se repite dentro del mismo camino.
    """

    def __init__(self, entorno, max_pasos=5_000_000):
        if not isinstance(max_pasos, int) or max_pasos <= 0:
            raise ValueError("max_pasos debe ser un entero positivo.")
        self.env = entorno
        self.max_pasos = max_pasos
        self._reiniciar()

    def _reiniciar(self):
        self.entrada = None
        self.salidas = {}
        self.pasos = 0
        self.expandidas = 0
        self.retrocesos = 0
        self.truncado = False
        self.motivo_truncado = None

    def resolver(self):
        self._reiniciar()
        entradas = self.env.entradas()
        if not entradas:
            return self._informe()
        self.entrada = entradas[0]

        ruta = [self.entrada]
        en_ruta = {self.entrada}
        pila = [iter(self.env.percibir(self.entrada))]
        self.expandidas = 1

        while pila:
            if self.pasos >= self.max_pasos:
                self.truncado = True
                self.motivo_truncado = "límite de pasos alcanzado"
                break

            try:
                d, tipo, destino = next(pila[-1])
                self.pasos += 1
            except StopIteration:
                pila.pop()
                en_ruta.remove(ruta.pop())
                self.retrocesos += 1
                continue

            if tipo == "borde":
                if not self._es_entrada_exterior(ruta[-1], d):
                    self._registrar_salida(ruta, d)
            elif destino not in en_ruta:
                ruta.append(destino)
                en_ruta.add(destino)
                pila.append(iter(self.env.percibir(destino)))
                self.expandidas += 1

        return self._informe()

    def _es_entrada_exterior(self, pos, direccion):
        return pos == self.entrada and direccion == 0

    def _registrar_salida(self, ruta, direccion):
        clave = (ruta[-1], direccion)
        self.salidas.setdefault(clave, []).append(list(ruta))

    def _informe(self):
        todos = [camino for caminos in self.salidas.values() for camino in caminos]
        mas_corto = min(todos, key=len) if todos else None
        return {
            "entrada": self.entrada,
            "salidas": dict(self.salidas),
            "total_caminos": len(todos),
            "mas_corto": mas_corto,
            "tiene_solucion": bool(todos),
            "pasos": self.pasos,
            "expandidas": self.expandidas,
            "retrocesos": self.retrocesos,
            "truncado": self.truncado,
            "completo": not self.truncado,
            "motivo_truncado": self.motivo_truncado,
        }


def imprimirConCamino(lab, filas, columnas, camino=None, entrada=None, salida=None):
    en_camino = set(camino or [])
    lineas = []
    for i in range(filas):
        borde, cuerpo = ["+"], []
        for j in range(columnas):
            c = lab[i][j]
            borde.append("---" if c.paredSuperior else "   ")
            borde.append("+")
            cuerpo.append("|" if c.paredIzquierda else " ")
            if (i, j) == entrada:        cuerpo.append(" S ")
            elif (i, j) == salida:       cuerpo.append(" E ")
            elif (i, j) in en_camino:    cuerpo.append(" * ")
            else:                        cuerpo.append("   ")
        cuerpo.append("|" if lab[i][columnas - 1].paredDerecha else " ")
        lineas += ["".join(borde), "".join(cuerpo)]
    base = ["+"]
    for j in range(columnas):
        base.append("---" if lab[filas - 1][j].paredInferior else "   ")
        base.append("+")
    lineas.append("".join(base))
    print("\n".join(lineas))


# ---- Ejecución: el agente recibe el laberinto del Bloque 1 ----
entorno = Entorno(lab, filas, columnas)
agente = AgenteLaberinto(entorno)
informe = agente.resolver()

print(f"Entrada detectada    : {informe['entrada']}")
print(f"Opciones evaluadas   : {informe['pasos']}")
print(f"Estados expandidos   : {informe['expandidas']}")
print(f"Retrocesos realizados: {informe['retrocesos']}")

if informe["truncado"]:
    print(f"Exploración incompleta: {informe['motivo_truncado']}.")

if not informe["tiene_solucion"] and informe["truncado"]:
    print("Resultado: INCONCLUSO; todavía podría existir una salida.\n")
    imprimirConCamino(lab, filas, columnas, entrada=informe["entrada"])
elif not informe["tiene_solucion"]:
    print("Resultado: NO existe camino a ninguna salida.\n")
    imprimirConCamino(lab, filas, columnas, entrada=informe["entrada"])
else:
    for (celda, d), caminos in informe["salidas"].items():
        print(f"Salida en {celda} (abertura {NOMBRES[d]}): {len(caminos)} camino(s), "
              f"el más corto mide {min(len(c) for c in caminos)} celdas")
    etiqueta = "Caminos encontrados" if informe["truncado"] else "Total de caminos"
    print(f"\n{etiqueta}: {informe['total_caminos']}")
    mc = informe["mas_corto"]
    print(f"Camino más corto ({len(mc)} celdas): {mc}\n")
    imprimirConCamino(lab, filas, columnas, mc, informe["entrada"], mc[-1])

Entrada detectada    : (0, 0)
Opciones evaluadas   : 981
Estados expandidos   : 487
Retrocesos realizados: 487
Salida en (14, 0) (abertura abajo): 2 camino(s), el más corto mide 31 celdas

Total de caminos: 2
Camino más corto (31 celdas): [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2), (2, 3), (2, 4), (1, 4), (1, 5), (1, 6), (2, 6), (3, 6), (4, 6), (5, 6), (5, 5), (6, 5), (7, 5), (7, 4), (7, 3), (8, 3), (9, 3), (10, 3), (10, 2), (10, 1), (11, 1), (12, 1), (12, 2), (13, 2), (13, 1), (14, 1), (14, 0)]

+   +---+---+---+---+---+---+---+---+---+---+---+---+---+---+
| S   * |   |       |   |               |   |   |   |       |
+---+   +   +---+   +   +   +---+---+   +   +   +   +   +---+
|   | *   *     | *   *   * |                       |       |
+   +---+   +---+   +   +   +---+---+---+---+   +   +   +   +
|   |   | *   *   * |   | *         |           |       |   |
+   +   +   +---+---+   +   +---+---+---+   +---+   +---+   +
|       |   |       |   | * |           |   |       |       |
+---+

EXPLORACIÓN POR FRONTERAS CON A*

In [30]:
# ==========================================================
# BLOQUE 3: EXPLORACIÓN ESTOCÁSTICA POR FRONTERAS
# A* solo navega hacia fronteras conocidas; nunca estima dónde está la salida.
# ==========================================================
import heapq
import random


class AgenteExploradorFronteras:
    """
    Construye un mapa únicamente con percepciones locales. Selecciona al azar
    entre las fronteras más cercanas y usa A* para llegar a la elegida.
    """

    def __init__(self, entorno, max_movimientos=5_000_000, semilla=None):
        if not isinstance(max_movimientos, int) or max_movimientos <= 0:
            raise ValueError("max_movimientos debe ser un entero positivo.")
        self.env = entorno
        self.max_movimientos = max_movimientos
        self.semilla = semilla
        self._reiniciar()

    def _reiniciar(self):
        self.rng = random.Random(self.semilla)
        self.entrada = None
        self.posicion = None
        self.salida = None
        self.mapa = {}
        self.visitadas = set()
        self.fronteras = set()
        self.trayectoria = []
        self.movimientos = 0
        self.replanificaciones = 0
        self.expandidas = 0
        self.truncado = False
        self.motivo_truncado = None

    def resolver(self):
        self._reiniciar()
        entradas = self.env.entradas()
        if not entradas:
            return self._informe(None)
        self.entrada = self.rng.choice(entradas)
        self.posicion = self.entrada
        self.trayectoria = [self.entrada]

        while True:
            direccion_salida = self._percibir_actual()
            if direccion_salida is not None:
                self.salida = (self.posicion, direccion_salida)
                return self._informe(self._ruta_conocida(self.entrada, self.posicion))
            if not self.fronteras:
                return self._informe(None)

            objetivo = self._seleccionar_frontera()
            ruta = self._ruta_conocida(self.posicion, objetivo)
            self.replanificaciones += 1
            if ruta is None:
                self.truncado = True
                self.motivo_truncado = "frontera inaccesible en el mapa conocido"
                return self._informe(None)
            if not self._seguir(ruta):
                return self._informe(None)

    def _percibir_actual(self):
        """Actualiza el mapa y devuelve la dirección de salida, si existe."""
        pos = self.posicion
        self.visitadas.add(pos)
        self.fronteras.discard(pos)
        self.mapa.setdefault(pos, set())

        salida = None
        for d, tipo, destino in self.env.percibir(pos):
            if tipo == "borde":
                if not (pos == self.entrada and d == 0):
                    salida = d
                continue
            self.mapa[pos].add(destino)
            self.mapa.setdefault(destino, set()).add(pos)
            if destino not in self.visitadas:
                self.fronteras.add(destino)
        return salida

    def _seleccionar_frontera(self):
        """Elige al azar entre las fronteras Manhattan más cercanas."""
        distancia = lambda p: self._h(self.posicion, p)
        minima = min(map(distancia, self.fronteras))
        candidatas = sorted(p for p in self.fronteras if distancia(p) == minima)
        return self.rng.choice(candidatas)

    @staticmethod
    def _h(origen, meta):
        return abs(origen[0] - meta[0]) + abs(origen[1] - meta[1])

    def _ruta_conocida(self, origen, meta):
        """A* sobre el mapa descubierto; la meta es una frontera conocida."""
        costos = {origen: 0}
        padres = {origen: None}
        abiertos = [(self._h(origen, meta), self.rng.random(), 0, origen)]

        while abiertos:
            _, _, costo, pos = heapq.heappop(abiertos)
            if costo != costos.get(pos):
                continue
            if pos == meta:
                ruta = []
                while pos is not None:
                    ruta.append(pos)
                    pos = padres[pos]
                return ruta[::-1]

            self.expandidas += 1
            for vecino in self.mapa.get(pos, ()):
                nuevo = costo + 1
                if nuevo < costos.get(vecino, float("inf")):
                    costos[vecino] = nuevo
                    padres[vecino] = pos
                    prioridad = nuevo + self._h(vecino, meta)
                    heapq.heappush(abiertos, (prioridad, self.rng.random(), nuevo, vecino))
        return None

    def _seguir(self, ruta):
        """Avanza hasta la primera celda aún no explorada de la ruta."""
        for destino in ruta[1:]:
            if self.movimientos >= self.max_movimientos:
                self.truncado = True
                self.motivo_truncado = "límite de movimientos alcanzado"
                return False
            self.posicion = destino
            self.trayectoria.append(destino)
            self.movimientos += 1
            if destino not in self.visitadas:
                break
        return True

    def _informe(self, camino):
        return {
            "entrada": self.entrada,
            "camino": camino,
            "trayectoria": list(self.trayectoria),
            "salida": self.salida,
            "tiene_solucion": self.salida is not None,
            "celdas_visitadas": len(self.visitadas),
            "movimientos": self.movimientos,
            "replanificaciones": self.replanificaciones,
            "expandidas": self.expandidas,
            "fronteras_pendientes": len(self.fronteras),
            "truncado": self.truncado,
            "motivo_truncado": self.motivo_truncado,
        }


# ---- Ejecución ----
entorno_f = Entorno(lab, filas, columnas)
inf = AgenteExploradorFronteras(entorno_f, semilla=0).resolver()

print(f"Entrada detectada       : {inf['entrada']}")
print(f"Celdas visitadas        : {inf['celdas_visitadas']}")
print(f"Movimientos realizados  : {inf['movimientos']}")
print(f"Replanificaciones       : {inf['replanificaciones']}")
print(f"Expansiones de A*        : {inf['expandidas']}")

if inf["truncado"]:
    print(f"Resultado: INCONCLUSO ({inf['motivo_truncado']}).\n")
    imprimirConCamino(lab, filas, columnas, entrada=inf["entrada"])
elif not inf["tiene_solucion"]:
    print("Resultado: NO existe camino a ninguna salida.\n")
    imprimirConCamino(lab, filas, columnas, entrada=inf["entrada"])
else:
    celda_salida, direccion = inf["salida"]
    print(f"Salida descubierta en {celda_salida} ({NOMBRES[direccion]})")
    print(f"Mejor ruta conocida ({len(inf['camino'])} celdas): {inf['camino']}\n")
    imprimirConCamino(lab, filas, columnas, inf["camino"], inf["entrada"], celda_salida)


Entrada detectada       : (0, 0)
Celdas visitadas        : 95
Movimientos realizados  : 188
Replanificaciones       : 94
Expansiones de A*        : 277
Salida descubierta en (14, 0) (abajo)
Mejor ruta conocida (31 celdas): [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2), (2, 3), (2, 4), (1, 4), (1, 5), (1, 6), (2, 6), (3, 6), (4, 6), (5, 6), (5, 5), (6, 5), (7, 5), (7, 4), (7, 3), (8, 3), (9, 3), (10, 3), (10, 2), (10, 1), (11, 1), (12, 1), (12, 2), (13, 2), (13, 1), (14, 1), (14, 0)]

+   +---+---+---+---+---+---+---+---+---+---+---+---+---+---+
| S   * |   |       |   |               |   |   |   |       |
+---+   +   +---+   +   +   +---+---+   +   +   +   +   +---+
|   | *   *     | *   *   * |                       |       |
+   +---+   +---+   +   +   +---+---+---+---+   +   +   +   +
|   |   | *   *   * |   | *         |           |       |   |
+   +   +   +---+---+   +   +---+---+---+   +---+   +---+   +
|       |   |       |   | * |           |   |       |       |
+---+   +---+---+   +